In [ ]:
# etl_unpivot_weather.py
# Unpivots ALL wide weather CSVs into tall format in one run.

import pandas as pd
from pathlib import Path

BASE = Path(r"C:\ProgramData\MySQL\MySQL Server 8.0\Uploads\energy_project_data")
OUT = BASE / "unpivot_all"
OUT.mkdir(exist_ok=True)

# All wide weather files you want to unpivot
WEATHER_FILES = [
    "temperature.csv",
    "pressure.csv",
    "humidity.csv",
    "wind_speed.csv",
    "wind_direction.csv",
    "weather_description.csv",
]

def unpivot_file(filename):
    src = BASE / filename
    df = pd.read_csv(src)

    # Identify ID columns vs hourly columns
    id_cols = ["city_name"]
    hour_cols = [c for c in df.columns if c not in id_cols]

    # Melt to tall format
    tall = df.melt(
        id_vars=id_cols,
        value_vars=hour_cols,
        var_name="datetime_utc",
        value_name="value"
    )

    # Split numeric vs text
    tall["value_text"] = tall["value"].astype(str)
    tall["value_num"] = pd.to_numeric(tall["value"], errors="coerce")

    tall["source_file"] = filename
    tall["measure"] = filename.replace(".csv", "")

    tall = tall[[
        "source_file",
        "measure",
        "datetime_utc",
        "city_name",
        "value_text",
        "value_num"
    ]]

    out_path = OUT / f"unpivot_{filename}"
    tall.to_csv(out_path, index=False)
    print(f"✓ Unpivoted {filename} → {out_path}")

def main():
    for f in WEATHER_FILES:
        unpivot_file(f)

if __name__ == "__main__":
    main()
